# HalfCheetah RL Competition — Colab Starter

Train and submit directly from the notebook. **Edit the config dict below** to change algorithm, phase, or hyperparameters, then run the training cell.
- **Phase 1:** `HalfCheetah-v4:run` — maximize forward velocity  
- **Phase 2:** `HalfCheetah-v4:backflip` — maximize angular velocity  
- **Phase 3:** `HalfCheetah-v4:efficient` — velocity per unit energy

In [ ]:
!pip install -q gymnasium[mujoco] stable-baselines3 pyyaml moviepy wandb

## Optional: W&B login

In [ ]:
# Uncomment to use Weights & Biases
# !pip install -q wandb
# import wandb
# wandb.login()

## 1. Set repo path and config

If you cloned the repo in Colab, set `REPO_DIR` to that path (e.g. `/content/lab-rl-aidl`). Then edit the **config** dict: change `env.id` for the phase, `algorithm` (ppo, sac, td3, a2c), or any hyperparameters.

In [ ]:
import sys
import os

# Repo root (if notebook is in src/, we go up one; else use cwd or set REPO_DIR manually)
REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "src" else os.getcwd()
# If you cloned to e.g. /content/lab-rl-aidl in Colab, set: REPO_DIR = "/content/lab-rl-aidl"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)  # so models/ and eval/seeds.json resolve correctly

# --- Edit this config: phase, algorithm, and hyperparameters ---
config = {
    "algorithm": "sac",           # one of: ppo, sac, td3, a2c
    "policy": "MlpPolicy",
    "env": {
        "id": "HalfCheetah-v4:run",   # or :backflip, :efficient
    },
    "hyperparameters": {
        "learning_rate": 3.0e-4,
        "buffer_size": 300_000,
        "batch_size": 256,
        "tau": 0.005,
        "gamma": 0.99,
        "train_freq": 1,
        "gradient_steps": 1,
        "ent_coef": "auto",
        "target_update_interval": 1,
    },
    "training": {
        "total_steps": 500_000,   # reduce for quick tests (e.g. 50_000)
        "seed": 42,
        "n_envs": 1,
        "verbose": 0,
        "model_path": "models/sac_agent.zip",
        "log_interval": 1000,
    },
    "evaluation": {
        "interval": 25_000,
        "episodes": 5,
        "video_interval": 100_000,
        "video_fps": 10,
    },
    "wandb": {"project": "aidl-rl-benchmark", "tags": ["training", "sac"]},
}

# PPO example (uncomment and set config = ppo_config to use):
# ppo_config = {**config, "algorithm": "ppo", "hyperparameters": {"learning_rate": 3e-4, "n_steps": 2048, "batch_size": 64, "n_epochs": 10, "gamma": 0.99, "gae_lambda": 0.95, "clip_range": 0.2, "ent_coef": 0.01, "vf_coef": 0.5, "max_grad_norm": 0.5}, "training": {**config["training"], "model_path": "models/ppo_agent.zip"}}


In [ ]:
from src.train import run_training

# Train (set use_wandb=True after wandb.login() in the optional cell above)
model_path = run_training(config, use_wandb=True)
print("Model saved at:", model_path)

## 2. Submit to the leaderboard

Set your session code, team name, and API URL, then run. Use the same `env_id` and `algo` as in your config.

In [ ]:
from src.eval_submit import evaluate_and_submit

# Edit these:
SESSION_CODE = "YOUR_SESSION_CODE"
TEAM_NAME = "Team Turbo"
API_URL = "https://YOUR-DOMAIN/api/submit"

# Model path and algo must match your config above
response = evaluate_and_submit(
    SESSION_CODE,
    TEAM_NAME,
    config["env"]["id"],
    config["algorithm"],
    str(model_path),
    API_URL,
    seeds_path=os.path.join(REPO_DIR, "eval", "seeds.json"),
    episodes_per_seed=5,
    use_wandb=True,
    eval_video=True,
    video_format="mp4",
)

## Tips
- **Change phase:** set `config["env"]["id"]` to `HalfCheetah-v4:backflip` or `HalfCheetah-v4:efficient`.
- **Try PPO or TD3:** set `config["algorithm"]` to `"ppo"` or `"td3"` and adjust `hyperparameters` (see `src/configs/*.yaml` in the repo for full examples).
- **Quick test:** set `config["training"]["total_steps"]` to e.g. `50_000` and re-run training.